# Q-learning (off-policy) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(z):
    z=np.asarray(z,dtype=float); e=np.exp(z-z.max()); return e/e.sum()
def discount(rewards,gamma=0.9):
    return sum((gamma**t)*r for t,r in enumerate(rewards))
def moving_avg(x,n=5):
    x=np.asarray(x,dtype=float); return np.convolve(x,np.ones(n)/n,mode='valid')
print('setup ok')

## Updating toward the best next action regardless of behavior
Q-learning separates how we explore from the greedy policy we want to learn.

In [ ]:
# 1) Monte Carlo return from a completed episode
rewards=np.array([0.,1.,0.,2.]); gamma=0.9
terms=np.array([(gamma**t)*r for t,r in enumerate(rewards)]); G=terms.sum()
plt.figure(figsize=(6,3)); plt.bar(range(len(terms)),terms); plt.title(f'complete-episode return = {G:.3f}')
assert abs(G-2.358)<1e-9

In [ ]:
# 2) TD error from a one-step target
gamma=0.9; r=1.; V_s=0.4; V_next=0.8; delta=r+gamma*V_next-V_s
plt.figure(figsize=(6,3)); plt.bar(['reward','discounted next','old value','TD error'],[r,gamma*V_next,V_s,delta])
plt.title('TD error = target - old estimate')
assert abs(delta-1.32)<1e-9

In [ ]:
# 3) eligibility trace assigns recent states more credit
gamma=0.9; lam=0.8; trace=np.array([(gamma*lam)**k for k in range(5)])
plt.figure(figsize=(6,3)); plt.bar(range(5),trace); plt.xlabel('steps back'); plt.ylabel('trace weight'); plt.title('recent visits receive stronger credit')
assert np.all(np.diff(trace)<0) and abs(trace[2]-0.5184)<1e-9

In [ ]:
# 4) SARSA and Q-learning targets differ when the exploratory next action is not greedy
Qnext=np.array([0.2,0.9]); r=0.5; gamma=0.9; actual_action=0
sarsa=r+gamma*Qnext[actual_action]; qlearn=r+gamma*Qnext.max()
plt.figure(figsize=(6,3)); plt.bar(['SARSA target','Q-learning target'],[sarsa,qlearn]); plt.ylim(0,1.5); plt.title('on-policy target is more cautious')
assert abs(sarsa-0.68)<1e-9 and abs(qlearn-1.31)<1e-9

In [ ]:
# 5) tabular action-value learning on a deterministic toy chain
Q=np.zeros((3,2)); alpha=0.5; gamma=0.9; transitions=[(0,1,0,1),(1,1,1,2),(2,0,0,2)]
for _ in range(8):
    for s,a,r,sp in transitions:
        Q[s,a]+=alpha*(r+gamma*Q[sp].max()-Q[s,a])
plt.figure(figsize=(5,3)); plt.imshow(Q,cmap='magma'); plt.colorbar(label='Q'); plt.xlabel('action'); plt.ylabel('state'); plt.title('learned action values')
assert Q[1,1]>Q[0,0] and Q[2,0]==0